# XaaS Capstone — Eurostat Trade Data Analysis
**Source:** Eurostat, Trade by partner country and enterprise size class (ext_tec10)

**Citation:** Eurostat (2026). International trade in goods — trade by enterprise characteristics (TEC). Dataset ext_tec10. European Commission.

**Purpose:** Analyze Lithuania's export patterns by enterprise size to validate the market opportunity for an Export-as-a-Service platform targeting SMBs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")
print("Libraries loaded")

## 1. Load Dataset

In [ ]:
# Load Eurostat ext_tec10 dataset
df = pd.read_csv("research/data/ext_tec10.csv.gz", compression='gzip')

# Filter: Lithuania, Exports, Thousand EUR
lt = df[(df['geo'] == 'LT') & (df['unit'] == 'THS_EUR') & (df['stk_flow'] == 'EXP')].copy()
lt['OBS_VALUE'] = pd.to_numeric(lt['OBS_VALUE'], errors='coerce')

print(f"Dataset: {len(lt)} rows")
print(f"Years: {sorted(lt['TIME_PERIOD'].unique())}")
print(f"Size classes: {list(lt['sizeclas'].unique())}")
print(f"Partner countries: {lt['partner'].nunique()}")

## 2. Lithuania Export Trends by Enterprise Size

In [ ]:
size_labels = {'LT10': '<10 employees (Micro)', '10-49': '10-49 (Small)',
               '50-249': '50-249 (Medium)', 'GE250': '250+ (Large)'}
colors = {'LT10': '#3498db', '10-49': '#2ecc71', '50-249': '#f39c12', 'GE250': '#e74c3c'}

fig, ax = plt.subplots(figsize=(12, 6))
for sc in ['LT10', '10-49', '50-249', 'GE250']:
    data = lt[(lt['sizeclas'] == sc)].groupby('TIME_PERIOD')['OBS_VALUE'].sum() / 1e6
    ax.plot(data.index, data.values, 'o-', label=size_labels[sc], color=colors[sc], linewidth=2)

ax.set_title("Lithuania Exports by Enterprise Size (2015-2024)", fontsize=14, fontweight='bold')
ax.set_xlabel("Year")
ax.set_ylabel("Exports (Billion EUR)")
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("research/reports/lt_exports_by_size.png", dpi=150)
plt.show()

## 3. SME Share of Exports

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2024 pie
latest = lt[lt['TIME_PERIOD'] == 2024]
sme = latest[latest['sizeclas'].isin(['LT10', '10-49', '50-249'])]['OBS_VALUE'].sum()
large = latest[latest['sizeclas'] == 'GE250']['OBS_VALUE'].sum()
axes[0].pie([sme, large], labels=['SMEs\n(<250 employees)', 'Large\n(250+)'],
            autopct='%1.1f%%', colors=['#3498db', '#e74c3c'], startangle=140,
            textprops={'fontsize': 13})
axes[0].set_title("SMEs vs Large Enterprises\nin Lithuania Exports (2024)", fontsize=13, fontweight='bold')

# SME share over time
years = sorted(lt['TIME_PERIOD'].unique())
sme_shares = []
for y in years:
    yr = lt[lt['TIME_PERIOD'] == y]
    s = yr[yr['sizeclas'].isin(['LT10', '10-49', '50-249'])]['OBS_VALUE'].sum()
    t = yr[yr['sizeclas'] == 'TOTAL']['OBS_VALUE'].sum()
    sme_shares.append(s / t * 100 if t > 0 else 0)

axes[1].bar(years, sme_shares, color='#3498db', alpha=0.8)
axes[1].set_title("SME Share of Lithuania Exports (%)", fontsize=13, fontweight='bold')
axes[1].set_ylabel("SME Export Share (%)")
axes[1].set_ylim(40, 60)
for i, v in enumerate(sme_shares):
    axes[1].text(years[i], v + 0.3, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig("research/reports/lt_sme_share.png", dpi=150)
plt.show()

print(f"\nSME export share 2024: {sme/(sme+large)*100:.1f}%")
print(f"SME exports: EUR {sme/1e6:.1f} billion")
print(f"Large exports: EUR {large/1e6:.1f} billion")

## 4. Top SME Export Destinations

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sme_2024 = lt[(lt['TIME_PERIOD'] == 2024) & (lt['sizeclas'].isin(['LT10', '10-49', '50-249']))]
sme_by_partner = sme_2024.groupby('Geopolitical entity (partner)')['OBS_VALUE'].sum().sort_values(ascending=False)
sme_by_partner = sme_by_partner[~sme_by_partner.index.str.contains('European|Euro|Extra|Intra|World|EFTA|Total', case=False, na=False)]
top15 = sme_by_partner.head(15) / 1e6

colors_bar = sns.color_palette("viridis", len(top15))
ax.barh(top15.index[::-1], top15.values[::-1], color=colors_bar)
ax.set_title("Top 15 Export Destinations for Lithuanian SMEs (2024)", fontsize=14, fontweight='bold')
ax.set_xlabel("Exports (Billion EUR)")
plt.tight_layout()
plt.savefig("research/reports/lt_sme_destinations.png", dpi=150)
plt.show()

print("\nTop 10 SME export destinations (2024):")
for country, val in top15.head(10).items():
    print(f"  {country}: EUR {val:.2f}B")

## 5. Export Growth by Enterprise Size (2019-2024)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
growth_data = {}
for sc in ['LT10', '10-49', '50-249', 'GE250']:
    y2019 = lt[(lt['TIME_PERIOD'] == 2019) & (lt['sizeclas'] == sc)]['OBS_VALUE'].sum()
    y2024 = lt[(lt['TIME_PERIOD'] == 2024) & (lt['sizeclas'] == sc)]['OBS_VALUE'].sum()
    growth = ((y2024 - y2019) / y2019) * 100
    growth_data[size_labels[sc]] = growth

bars = ax.bar(growth_data.keys(), growth_data.values(),
              color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'])
ax.set_title("Export Growth 2019 to 2024 by Enterprise Size", fontsize=14, fontweight='bold')
ax.set_ylabel("Growth (%)")
for bar, val in zip(bars, growth_data.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%',
            ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("research/reports/lt_export_growth.png", dpi=150)
plt.show()

## 6. Market Diversification by Enterprise Size

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for sc, label, color in [('LT10', 'Micro (<10)', '#3498db'), ('10-49', 'Small (10-49)', '#2ecc71'),
                           ('50-249', 'Medium (50-249)', '#f39c12'), ('GE250', 'Large (250+)', '#e74c3c')]:
    counts = []
    for y in years:
        yr_data = lt[(lt['TIME_PERIOD'] == y) & (lt['sizeclas'] == sc)]
        yr_data = yr_data[~yr_data['Geopolitical entity (partner)'].str.contains('European|Euro|Extra|Intra|World|EFTA|Total', case=False, na=False)]
        n_countries = (yr_data['OBS_VALUE'] > 0).sum()
        counts.append(n_countries)
    ax.plot(years, counts, 'o-', label=label, color=color, linewidth=2)

ax.set_title("Number of Export Markets by Enterprise Size", fontsize=14, fontweight='bold')
ax.set_ylabel("Number of Destination Countries")
ax.set_xlabel("Year")
ax.legend()
plt.tight_layout()
plt.savefig("research/reports/lt_market_diversification.png", dpi=150)
plt.show()

## 7. Key Findings Summary

In [ ]:
print("=" * 60)
print("KEY FINDINGS — MARKET VALIDATION FOR XaaS PLATFORM")
print("=" * 60)

total_2024 = lt[lt['TIME_PERIOD'] == 2024].groupby('sizeclas')['OBS_VALUE'].sum()
sme_total = total_2024[['LT10', '10-49', '50-249']].sum()
all_total = total_2024['TOTAL']

print(f"\n1. MARKET SIZE")
print(f"   Lithuania total exports 2024: EUR {all_total/1e6:.1f} billion")
print(f"   SME exports: EUR {sme_total/1e6:.1f} billion ({sme_total/all_total*100:.1f}% of total)")

print(f"\n2. TARGET SEGMENT BREAKDOWN")
for sc, label in [('LT10','Micro <10'), ('10-49','Small 10-49'), ('50-249','Medium 50-249')]:
    val = total_2024[sc]
    print(f"   {label}: EUR {val/1e6:.1f}B ({val/all_total*100:.1f}%)")

print(f"\n3. GROWTH TREND")
for sc, label in [('LT10','Micro'), ('10-49','Small'), ('50-249','Medium'), ('GE250','Large')]:
    y19 = lt[(lt['TIME_PERIOD']==2019)&(lt['sizeclas']==sc)]['OBS_VALUE'].sum()
    y24 = lt[(lt['TIME_PERIOD']==2024)&(lt['sizeclas']==sc)]['OBS_VALUE'].sum()
    print(f"   {label}: {((y24-y19)/y19)*100:.1f}% growth (2019-2024)")

print(f"\n4. IMPLICATION FOR XaaS PLATFORM")
print(f"   - SMEs represent over half of Lithuania's exports")
print(f"   - Micro enterprises (<10 employees) are growing fastest")
print(f"   - These are exactly the companies that lack in-house export capabilities")
print(f"   - Even capturing 1% of SME export value = EUR {sme_total/1e6*0.01:.1f}B addressable market")